In [1]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate_hkqai").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"new_dataset/{data_set}.json") as f:
    json_data = json.load(f)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict_ccpvdz"]
    # full_subset_dict = json.load(f)["full_subset_dict_test"]

data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for data_path in data_path_list:
        data = pd.read_csv(data_path)
        data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
        data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
        data_dft = data["dft_ene"].to_numpy() * 627.5094733748099
        data_scf = data["scf_ene"].to_numpy() * 627.5094733748099
        data_cc = data["cc_ene"].to_numpy() * 627.5094733748099

        data_error_scf_ele = data["error_scf_ele"].to_numpy()
        data_error_dft_ele = data["error_dft_ele"].to_numpy()
        data_error_scf_dip = data["error_scf_dip"].to_numpy()
        data_error_dft_dip = data["error_dft_dip"].to_numpy()
        data_subset[f"{data_path_name}_summary"] = {
            "error_scf_ele": data_error_scf_ele,
            "error_dft_ele": data_error_dft_ele,
            "error_scf_dip": data_error_scf_dip,
            "error_dft_dip": data_error_dft_dip,
        }

        if "delta_d3bj" in data.columns:
            data_d3bj = data["delta_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_d3bj = np.zeros_like(data_dft)
        if "delta_d3zero" in data.columns:
            data_d3zero = data["delta_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_d3zero = np.zeros_like(data_dft)

        if "modified_dft_d3bj" in data.columns:
            data_dft_d3bj = data["modified_dft_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3bj = data_d3bj
        if "modified_dft_d3zero" in data.columns:
            data_dft_d3zero = data["modified_dft_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_dft_d3zero = data_d3zero
        if "modified_ai_d3bj" in data.columns:
            data_ai_d3bj = data["modified_ai_d3bj"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3bj = data_d3bj
        if "modified_ai_d3zero" in data.columns:
            data_ai_d3zero = data["modified_ai_d3zero"].to_numpy() * 627.5094733748099
        else:
            data_ai_d3zero = data_d3zero

        del data_d3bj, data_d3zero

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "dft_d3bj": [],
                "dft_d3zero": [],
                "ai": [],
                "ai_d3bj": [],
                "ai_d3zero": [],
                "cc": [],
                "error_scf_ele": [],
                "error_dft_ele": [],
                "error_scf_dip": [],
                "error_dft_dip": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(
                            f"Warning: {i_molecule_name} not found in {data_path.stem} data file"
                        )
                    continue
                data_subset[name_subset]["error_scf_ele"].append(
                    data_error_scf_ele[col[0]]
                )
                data_subset[name_subset]["error_dft_ele"].append(
                    data_error_dft_ele[col[0]]
                )
                data_subset[name_subset]["error_scf_dip"].append(
                    data_error_scf_dip[col[0]]
                )
                data_subset[name_subset]["error_dft_dip"].append(
                    data_error_dft_dip[col[0]]
                )

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_dft_d3bj = 0
                atomic_energy_dft_d3zero = 0
                atomic_energy_ai = 0
                atomic_energy_ai_d3bj = 0
                atomic_energy_ai_d3zero = 0
                atomic_energy_cc = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                        atomic_energy_dft_d3bj += data_dft_d3bj[col[0]] * stoichiometry
                        atomic_energy_dft_d3zero += (
                            data_dft_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_ai += data_scf[col[0]] * stoichiometry
                        atomic_energy_ai_d3bj += data_ai_d3bj[col[0]] * stoichiometry
                        atomic_energy_ai_d3zero += (
                            data_ai_d3zero[col[0]] * stoichiometry
                        )
                        atomic_energy_cc += data_cc[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["dft_d3bj"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3bj
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["dft_d3zero"].append(
                        abs(
                            atomic_energy_dft
                            + atomic_energy_dft_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["ai"].append(
                        abs(atomic_energy_ai - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3bj"].append(
                        abs(atomic_energy_ai + atomic_energy_ai_d3bj - atomic_energy_cc)
                    )
                    data_subset[name_subset]["ai_d3zero"].append(
                        abs(
                            atomic_energy_ai
                            + atomic_energy_ai_d3zero
                            - atomic_energy_cc
                        )
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_ai = np.argsort(data_subset[name_subset]["ai"])[
                    ::-1
                ][:5]
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ][:5]
                for i in range(len(argsort_atomic_energy_ai)):
                    i_reaction_name = data_subset[name_subset]["name"][
                        argsort_atomic_energy_ai[i]
                    ]
                    i_reaction = json_data[f"reaction-{i_subset}"][i_reaction_name]
                    print(
                        f"Top {i+1} AI: {data_subset[name_subset]['ai'][argsort_atomic_energy_ai[i]]} kcal/mol, {i_reaction_name} in {name_subset}",
                    )
                    systems_list = i_reaction["systems"]
                    stoichiometry_list = i_reaction["stoichiometry"]
                    for j in range(len(systems_list)):
                        mole_name = (
                            systems_list[j]
                            if i_subset == "BH76RC"
                            else f"{i_subset}-{systems_list[j]}"
                        )
                        stoichiometry = int(stoichiometry_list[j])
                        if mole_name in json_data:
                            if isinstance(json_data[mole_name], str):
                                mole_name = json_data[mole_name]
                        print(f"  {stoichiometry} * {mole_name}", end="")
                        col = np.where(data_name == mole_name)[0]
                        if col.size == 1:
                            error_energy_ai = data_scf[col[0]] - data_cc[col[0]]
                            print(f"  {stoichiometry} * {error_energy_ai}", end="")
                    print()

                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

data_path_name_list = [
    data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    for data_path in data_path_list
]
header = pd.MultiIndex.from_product(
    [
        data_path_name_list,
        [
            "AI",
            "DFT",
            "AI_D3BJ",
            "DFT_D3BJ",
            "AI_D3ZERO",
            "DFT_D3ZERO",
            "Processed",
        ],
    ],
    names=["data_path", "Disp type"],
)

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

df_summary_subset_ele = pd.DataFrame(
    columns=pd.MultiIndex.from_product(
        [
            data_path_name_list,
            [
                "error_scf_ele",
                "error_dft_ele",
                "error_scf_dip",
                "error_dft_dip",
            ],
        ],
        names=["data_path", "Ele type"],
    )
)

for data_path in data_path_list:
    mean_absolute_deviation_list = []
    data_path_name = data_path.stem.split("_atom-1-")[1].split("_gmtkn")[0]
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_ele")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_ele"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_scf_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_scf_dip"]
    )
    df_summary_subset_ele.loc["summary", (data_path_name, "error_dft_dip")] = np.mean(
        data_subset[f"{data_path_name}_summary"]["error_dft_dip"]
    )

    for name_set, subset_list_ in full_subset_dict.items():
        subset_ai = {}
        wtmad_1_ai = {}
        wtmad_2_ai = {}
        subset_dft = {}
        wtmad_1_dft = {}
        wtmad_2_dft = {}

        for d3_name in ["", "_d3bj", "_d3zero"]:
            subset_ai[d3_name] = []
            wtmad_1_ai[d3_name] = []
            wtmad_2_ai[d3_name] = []
            subset_dft[d3_name] = []
            wtmad_1_dft[d3_name] = []
            wtmad_2_dft[d3_name] = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{data_path_name}_{i_subset}"
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_ele")] = (
                np.mean(data_subset[name_subset]["error_scf_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_ele")] = (
                np.mean(data_subset[name_subset]["error_dft_ele"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_scf_dip")] = (
                np.mean(data_subset[name_subset]["error_scf_dip"])
            )
            df_summary_subset_ele.loc[i_subset, (data_path_name, "error_dft_dip")] = (
                np.mean(data_subset[name_subset]["error_dft_dip"])
            )

            if len(data_subset[name_subset]["ai"]) == 0:
                for col_name in [
                    "AI",
                    "DFT",
                    "AI_D3BJ",
                    "DFT_D3BJ",
                    "AI_D3ZERO",
                    "DFT_D3ZERO",
                ]:
                    df_summary_subset.loc[i_subset, (data_path_name, col_name)] = 0
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                for d3_name in ["", "_d3bj", "_d3zero"]:
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"AI{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"ai{d3_name}"])
                    df_summary_subset.loc[
                        i_subset, (data_path_name, f"DFT{d3_name.upper()}")
                    ] = np.mean(data_subset[name_subset][f"dft{d3_name}"])
                df_summary_subset.loc[i_subset, (data_path_name, "Processed")] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["ai"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["ai"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['ai'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                for d3_name in ["", "_d3bj", "_d3zero"]:
                    subset_ai[d3_name] = np.append(
                        subset_ai[d3_name], data_subset[name_subset][f"ai{d3_name}"]
                    )
                    subset_dft[d3_name] = np.append(
                        subset_dft[d3_name], data_subset[name_subset][f"dft{d3_name}"]
                    )
                    wtmad_1_ai[d3_name] = np.append(
                        wtmad_1_ai[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"ai{d3_name}"]),
                    )
                    wtmad_1_dft[d3_name] = np.append(
                        wtmad_1_dft[d3_name],
                        wtmad_1 * np.mean(data_subset[name_subset][f"dft{d3_name}"]),
                    )
                    wtmad_2_ai[d3_name] = np.append(
                        wtmad_2_ai[d3_name],
                        data_subset[name_subset][f"ai{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                    wtmad_2_dft[d3_name] = np.append(
                        wtmad_2_dft[d3_name],
                        data_subset[name_subset][f"dft{d3_name}"]
                        / np.mean(data_subset[name_subset]["cc"]),
                    )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["ai"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)
        for d3_name in ["", "_d3bj", "_d3zero"]:
            mean_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(subset_ai[d3_name])
            )
            mean_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(subset_dft[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.mean(wtmad_1_ai[d3_name])
            )
            wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.mean(wtmad_1_dft[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                np.sum(wtmad_2_ai[d3_name])
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                np.sum(wtmad_2_dft[d3_name])
            )
        mean_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, (data_path_name, "Processed")] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = np.mean(mean_absolute_deviation_list) / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {data_path_name}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        for d3_name in ["", "_d3bj", "_d3zero"]:
            wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            )
            wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")] = (
                mean_absolute_deviation
                * wtmad_2_subset.loc[
                    name_set, (data_path_name, f"DFT{d3_name.upper()}")
                ]
            )

    wtmad_1_subset.loc["summary", (data_path_name, "Processed")] = "--"
    wtmad_2_subset.loc["summary", (data_path_name, "Processed")] = "--"
    for d3_name in ["", "_d3bj", "_d3zero"]:
        wtmad_1_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_1_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"AI{d3_name.upper()}")] = 0
        wtmad_2_subset.loc["summary", (data_path_name, f"DFT{d3_name.upper()}")] = 0
        for name_set in full_subset_dict.keys():
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_1_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_1_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"AI{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"AI{d3_name.upper()}")]
            wtmad_2_subset.loc[
                "summary", (data_path_name, f"DFT{d3_name.upper()}")
            ] += wtmad_2_subset.loc[name_set, (data_path_name, f"DFT{d3_name.upper()}")]

# print("Summary")
# display(df_summary_subset_ele)
print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# save summary to csv with date
df_summary_subset_ele.to_csv(f"../validate/df_summary_subset_ele_{date}.csv")
df_summary_subset.to_csv(f"../validate/summary_subset_{date}.csv")
mean_subset.to_csv(f"../validate/mean_subset_{date}.csv")
wtmad_1_subset.to_csv(f"../validate/wtmad_1_subset_{date}.csv")
wtmad_2_subset.to_csv(f"../validate/wtmad_2_subset_{date}.csv")
# save summary to excel with date
df_summary_subset_ele.to_excel(f"../validate/df_summary_subset_ele_{date}.xlsx")
df_summary_subset.to_excel(f"../validate/summary_subset_{date}.xlsx")
mean_subset.to_excel(f"../validate/mean_subset_{date}.xlsx")
wtmad_1_subset.to_excel(f"../validate/wtmad_1_subset_{date}.xlsx")
wtmad_2_subset.to_excel(f"../validate/wtmad_2_subset_{date}.xlsx")

cc-pVDZ
Top 1 AI: 38.4684967326466 kcal/mol, 133 in 3036943_W4_11
  -1 * W4_11-b2  -1 * 10.333999411293917  2 * W4_11-b  2 * 0.00735858251391619
Top 2 AI: 32.5134282814106 kcal/mol, 128 in 3036943_W4_11
  -1 * W4_11-s4-c2v  -1 * -13.62584772054106  4 * W4_11-s  4 * -0.08260939861065708
Top 3 AI: 26.098701825831085 kcal/mol, 131 in 3036943_W4_11
  -1 * W4_11-oclo  -1 * -32.23040660680272  2 * W4_11-o  2 * -0.01763694222609047  1 * W4_11-cl  1 * 0.3182955590891652
Top 4 AI: 22.529547434707638 kcal/mol, 104 in 3036943_W4_11
  -1 * W4_11-hnnn  -1 * -12.704742801826796  1 * W4_11-h  1 * -0.12572445527990794  3 * W4_11-n  3 * -0.08974874942941824
Top 5 AI: 21.856567947237636 kcal/mol, 129 in 3036943_W4_11
  -1 * W4_11-of  -1 * -12.823814730218146  1 * W4_11-o  1 * -0.01763694222609047  1 * W4_11-f  1 * -0.10107343958225101
Top 1 DFT: 64.9391765203327 kcal/mol
Top 2 DFT: 64.82811637483246 kcal/mol
Top 3 DFT: 58.7866408124537 kcal/mol
Top 4 DFT: 55.610892266675364 kcal/mol
Top 5 DFT: 55.139185

/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packa

MAE


data_path    3036943                                                       \
Disp type         AI        DFT   AI_D3BJ   DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1        6.118434  18.203799  6.459284  18.914367  5.991956  18.232302   
sub2       11.078233   7.778315  7.385627  11.988613   6.70284   7.600269   
sub3             NaN        NaN       NaN        NaN       NaN        NaN   
sub4             NaN        NaN       NaN        NaN       NaN        NaN   
sub5        0.978262   0.574513  0.409574    0.54405  0.386263   0.565887   

data_path             1424849                       ...                       \
Disp type Processed        AI        DFT   AI_D3BJ  ... AI_D3ZERO DFT_D3ZERO   
sub1         8 / 18  2.002477  13.710857  2.341161  ...  1.840461  13.713797   
sub2          0 / 7  5.510109   6.612164  9.257621  ...  6.192242   6.757732   
sub3          0 / 7  2.617318   6.261376  3.548881  ...  3.228945   6.934991   
sub4         0 / 11   1.93524   3.272197  2.924356  ...  2.900361   4.988172   
sub5          0 / 8  1.626727   1.321288   1.52729  ...  1.545785   0.872793   

data_path             1513512                                             \
Disp type Processed        AI        DFT    AI_D3BJ   DFT_D3BJ AI_D3ZERO   
sub1           DONE  6.282354  13.717062   6.868466  14.265974  6.365582   
sub2           DONE  7.536765   6.612828  11.817093   9.131704  8.930029   
sub3           DONE  4.620451    6.26131   5.733397   7.356016  5.317836   
sub4           DONE   2.38953   3.272965   3.818374   5.029019  3.790518   
sub5           DONE  1.250183   1.322771   0.717525   0.928201  0.665034   

data_path                       
Disp type DFT_D3ZERO Processed  
sub1       13.720054      DONE  
sub2        6.760778      DONE  
sub3        6.935442      DONE  
sub4        4.986451      DONE  
sub5        0.872943      DONE  

[5 rows x 21 columns]

wtmad_1


data_path    3036943                                                      \
Disp type         AI        DFT   AI_D3BJ  DFT_D3BJ AI_D3ZERO DFT_D3ZERO   
sub1        5.264198   6.676977  4.441266  6.613591  4.480294   6.300322   
sub2       25.373279  24.561859  4.058777   4.88558  8.833039   8.581265   
sub3             NaN        NaN       NaN       NaN       NaN        NaN   
sub4             NaN        NaN       NaN       NaN       NaN        NaN   
sub5        9.782615   5.745127  4.095739  5.440497  3.862631   5.658872   
summary          NaN        NaN       NaN       NaN       NaN        NaN   

data_path              1424849                        ...             \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1         8 / 18   3.239113   7.420922   2.966668  ...   2.651677   
sub2          0 / 7  13.142366   12.53052  11.927684  ...  11.524607   
sub3          0 / 7   4.832263   6.984991   5.854163  ...   5.522858   
sub4         0 / 11  11.556831  11.029773  13.901938  ...  12.876599   
sub5          0 / 8  15.335367  11.571826  14.145431  ...  14.544885   
summary          --  48.105939  49.538032  48.795883  ...  47.120625   

data_path                         1513512                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        7.102094      DONE   5.917804   7.427913   6.173631   7.422353   
sub2       10.005295      DONE   9.677109  12.530346   7.731942   9.967392   
sub3        7.521852      DONE   5.362819   6.982875   6.557344   8.149361   
sub4       13.798614      DONE  10.472747  11.046974  12.507724  14.856703   
sub5        7.523212      DONE  10.880025  11.599184   6.460544   8.076713   
summary    45.951067        --  42.310504  49.587291  39.431184  48.472523   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        5.786012   7.109349      DONE  
sub2        7.407743  10.004803      DONE  
sub3        5.935941   7.519994      DONE  
sub4       11.478243  13.779798      DONE  
sub5        5.881129   7.508553      DONE  
summary    36.489069  45.922496        --  

[6 rows x 21 columns]

wtmad_2


data_path    3036943                                                        \
Disp type         AI        DFT    AI_D3BJ  DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
sub1        14.00995  26.992259  13.326963  27.45632  13.062257  26.722526   
sub2         6.06036   5.873226   0.956717  1.144589   2.103308   2.040891   
sub3             0.0        0.0        0.0       0.0        0.0        0.0   
sub4             0.0        0.0        0.0       0.0        0.0        0.0   
sub5        1.552508   0.911756   0.649997  0.863411   0.613002   0.898067   
summary    21.622818   33.77724  14.933677  29.46432  15.778567  29.661483   

data_path              1424849                        ...             \
Disp type Processed         AI        DFT    AI_D3BJ  ...  AI_D3ZERO   
sub1         8 / 18   1.438234   4.040696   1.379994  ...   1.263846   
sub2          0 / 7   3.339073   3.635331   2.970755  ...   2.735086   
sub3          0 / 7   1.303233   2.611064    1.66299  ...   1.549133   
sub4         0 / 11   4.683102   4.159493   6.811273  ...   6.578153   
sub5          0 / 8   6.331811    4.66236    6.30221  ...   6.387196   
summary          --  17.095452  19.108945  19.127222  ...  18.513414   

data_path                         1513512                                   \
Disp type DFT_D3ZERO Processed         AI        DFT    AI_D3BJ   DFT_D3BJ   
sub1        3.989769      DONE   2.668675   4.045306   2.808633   4.115977   
sub2        2.545688      DONE   3.206768   3.637148   2.047904   2.377001   
sub3        2.873647      DONE   1.989474   2.610843    2.42163   3.039958   
sub4         6.62018      DONE   4.130245    4.16366   6.081695   6.850967   
sub5         3.19725      DONE   4.368745   4.670544   2.805652    3.50597   
summary    19.226535        --  16.363907  19.127501  16.165515  19.889873   

data_path                                  
Disp type  AI_D3ZERO DFT_D3ZERO Processed  
sub1        2.675278   3.994343      DONE  
sub2        2.176739   2.547486      DONE  
sub3        2.256968   2.873547      DONE  
sub4        5.851345   6.614613      DONE  
sub5        2.496356   3.192324      DONE  
summary    15.456686  19.222314        --  

[6 rows x 21 columns]

Summary of Subset
MAE


data_path    3036943                                                         \
Disp type         AI        DFT    AI_D3BJ   DFT_D3BJ  AI_D3ZERO DFT_D3ZERO   
W4_11       6.849464  29.111712   8.159832  30.739374   7.086307  29.426911   
G21EA       2.581223   9.754522   2.575859   9.751745   2.592166   9.757546   
G21IP       1.938484   8.953334    1.94495   8.946699   1.941726     8.9519   
DIPCS10     4.985363  12.310852    4.96582  12.334099    4.96276  12.264655   
PA26        2.360893   2.222399    2.61935   1.834736   2.489014   1.987376   
SIE4x4     18.489931  21.908516  18.919228  22.337814  18.921342  22.339928   
ALKBDE10   11.415343  18.125246  12.103946  18.838948  11.425615  18.135588   
YBDE18      5.891732   7.984356   4.727342   7.507393   4.930792   7.560119   
AL2X6       4.917457   5.659145   2.368365   1.332003    1.38317   1.829895   
HEAVYSB11   5.957114   5.391476   5.287455   8.010474   4.527058   5.984154   
NBPRC       4.906963   1.640573   3.525889   3.024418   3.587284   2.174489   
ALK8        7.397188   4.400063   2.018111   3.174803   4.748123   2.559066   
RC21               0          0          0          0          0          0   
G2RC               0          0          0          0          0          0   
BH76RC             0          0          0          0          0          0   
FH51               0          0          0          0          0          0   
TAUT15             0          0          0          0          0          0   
DC13               0          0          0          0          0          0   
MB16_43    21.441555  12.767682  18.652963  30.855364  15.182943  17.701997   
DARC               0          0          0          0          0          0   
RSE43              0          0          0          0          0          0   
BSR36        4.86024   4.784695   0.625226   0.668562   1.614778   1.539233   
CDIE20             0          0          0          0          0          0   
ISO34              0          0          0          0          0          0   
PArel              0          0          0          0          0          0   
BH76               0          0          0          0          0          0   
BHPERI             0          0          0          0          0          0   
BHDIV10            0          0          0          0          0          0   
INV24              0          0          0          0          0          0   
BHROT27            0          0          0          0          0          0   
PX13               0          0          0          0          0          0   
WCPT18             0          0          0          0          0          0   
RG18               0          0          0          0          0          0   
ADIM6              0          0          0          0          0          0   
S22                0          0          0          0          0          0   
S66                0          0          0          0          0          0   
WATER27            0          0          0          0          0          0   
CARBHB12           0          0          0          0          0          0   
PNICO23            0          0          0          0          0          0   
HAL59              0          0          0          0          0          0   
AHB21              0          0          0          0          0          0   
CHB6               0          0          0          0          0          0   
IL16               0          0          0          0          0          0   
IDISP              0          0          0          0          0          0   
ICONF              0          0          0          0          0          0   
ACONF              0          0          0          0          0          0   
Amino20x4   0.978262   0.574513   0.409574    0.54405   0.386263   0.565887   
PCONF21            0          0          0          0          0          0   
MCONF              0          0          0        

In [5]:
16.363907 / 19.127501, 12.60223 / 15.936448, 14.245033 / 17.35668

(0.8555172471301924, 0.7907803545683455, 0.8207233756686185)